In [1]:
# ===========================
# 0) IMPORTS
# ===========================
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl
import pandas as pd
import re


# ===========================
# 1) CONFIG (YOUR REAL COLUMNS)
# ===========================
# Owner (appraisal/current owner) file columns
APP_ACCT_COL      = "acct"
APP_OWNER_COL     = "name"
APP_ADDR_COL      = "site_address"
APP_PURCHASE_COL  = "purchase_date"     # will be parsed to datetime

# Deed file columns (as seen in your printout)
DEED_FILE_NO_COL  = "File No"
DEED_NAME_COL     = "Name"
DEED_SUBDIV_COL   = "SubDivision"
DEED_SECTION_COL  = "Section"
DEED_LOT_COL      = "Lot"
DEED_BLOCK_COL    = "Block"
DEED_UNIT_COL     = "Unit"
DEED_FILEDATE_COL = "FileDate"


# ===========================
# 2) NAME / ADDRESS HELPERS
# ===========================
ENTITY_TOKENS = {
    "LLC","INC","CORP","CO","COMPANY","LP","LTD","PLC","PC",
    "BANK","NA","N.A","N.A.","ASSOCIATION","MORTGAGE","SERVICING",
    "FUND","INVEST","INVESTMENTS","HOLDINGS","CAPITAL","PROPERTIES",
    "TRUST","TRUSTEE","ESTATE","HOA","ASSN","ASSOCIATES","CREDIT","UNION"
}
NOISE_TOKENS = {"ET","AL","ETAL","C/O","CARE","OF","THE","A","AN"}

STREET_SUFFIX_MAP = {
    "STREET":"ST","AVENUE":"AVE","ROAD":"RD","DRIVE":"DR","LANE":"LN","COURT":"CT",
    "BOULEVARD":"BLVD","PLACE":"PL","CIRCLE":"CIR","TERRACE":"TER","PARKWAY":"PKWY",
    "HIGHWAY":"HWY","TRAIL":"TRL","WAY":"WAY"
}
DIRECTION_MAP = {
    "NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W",
    "NORTHEAST":"NE","NORTHWEST":"NW","SOUTHEAST":"SE","SOUTHWEST":"SW"
}

def _upper_clean(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).upper().strip()
    s = re.sub(r"[^\w\s#]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_name(name: str) -> str:
    s = _upper_clean(name)
    if not s:
        return ""
    s = s.replace(" N A ", " NA ").replace(" N.A ", " NA ").replace(" N.A. ", " NA ")
    toks = [t for t in s.split() if t not in NOISE_TOKENS]
    return " ".join(toks)

def is_entity(name_norm: str) -> int:
    if not name_norm:
        return 0
    toks = set(name_norm.split())
    return int(any(t in toks for t in ENTITY_TOKENS))

def split_person_name(name_norm: str):
    """
    Your files are LAST FIRST MID (no comma). Return (first, mid, last).
    For org names, this is imperfect but still gives stable tokens.
    """
    if not name_norm:
        return ("", "", "")
    toks = name_norm.split()
    if len(toks) == 1:
        return ("", "", toks[0])
    last = toks[0]
    first = toks[1] if len(toks) >= 2 else ""
    mid = toks[2] if len(toks) >= 3 else ""
    if len(mid) > 1:
        mid = mid[0]
    return (first, mid, last)

def normalize_partial_address(addr: str) -> str:
    s = _upper_clean(addr)
    if not s:
        return ""
    toks = []
    for t in s.split():
        t = DIRECTION_MAP.get(t, t)
        t = STREET_SUFFIX_MAP.get(t, t)
        toks.append(t)
    return " ".join(toks).strip()

def parse_house_and_street(addr_norm: str):
    if not addr_norm:
        return ("", "")
    m = re.match(r"^(\d+)\s+(.*)$", addr_norm)
    if not m:
        return ("", addr_norm[:40].strip())
    house = m.group(1)
    rest = m.group(2).strip()
    rest = re.split(r"\b(APT|UNIT|STE|SUITE|#)\b", rest)[0].strip()
    core = " ".join(rest.split()[:4])
    return (house, core)


# ===========================
# 3) FEATURE BUILDERS
# ===========================
def build_features_appraisal(df_owner: pd.DataFrame) -> pd.DataFrame:
    df = df_owner.copy()
    df.columns = df.columns.str.strip()

    # keep raw fields for later export
    df["name_raw"] = df[APP_OWNER_COL].astype("string").str.strip()
    df["site_address_raw"] = df[APP_ADDR_COL].astype("string").str.strip()
    df["purchase_date_dt"] = pd.to_datetime(df[APP_PURCHASE_COL], errors="coerce")

    # name features used for matching
    df["name_norm"] = df["name_raw"].map(normalize_name)
    df["entity_flag"] = df["name_norm"].map(is_entity)

    fml = df["name_norm"].map(split_person_name)
    df["first_name"] = [x[0] for x in fml]
    df["mid_init"]   = [x[1] for x in fml]
    df["last_name"]  = [x[2] for x in fml]
    df["last3"]      = df["last_name"].str[:3].fillna("")

    # optional address features (not required, but nice)
    df["addr_norm"] = df["site_address_raw"].map(normalize_partial_address)
    hs = df["addr_norm"].map(parse_house_and_street)
    df["house_num"]   = [x[0] for x in hs]
    df["street_core"] = [x[1] for x in hs]
    df["street3"]     = df["street_core"].str.replace(" ", "", regex=False).str[:3].fillna("")

    return df

def build_features_deed(df_deed: pd.DataFrame) -> pd.DataFrame:
    df = df_deed.copy()
    df.columns = df.columns.str.strip()

    df["name_raw"] = df[DEED_NAME_COL].astype("string").str.strip()
    df["name_norm"] = df["name_raw"].map(normalize_name)
    df["entity_flag"] = df["name_norm"].map(is_entity)

    fml = df["name_norm"].map(split_person_name)
    df["first_name"] = [x[0] for x in fml]
    df["mid_init"]   = [x[1] for x in fml]
    df["last_name"]  = [x[2] for x in fml]
    df["last3"]      = df["last_name"].str[:3].fillna("")

    if DEED_FILEDATE_COL in df.columns:
        df["file_date_dt"] = pd.to_datetime(df[DEED_FILEDATE_COL], errors="coerce")

    return df


# ===========================
# 4) LOAD FILES + CLEAN
# ===========================
# --- Deed CSV (dtype=str keeps all fields as strings) ---
df_deed = pd.read_csv(
    r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\Deed transfer test data 08.08.25\Merged data ready for import 08.08.25\final_merged_buyers_only_08.08.25.csv",
    dtype=str,
    engine="python"
)

df_deed.columns = df_deed.columns.str.strip()
for c in df_deed.columns:
    df_deed[c] = df_deed[c].astype("string").str.strip()

# drop junk names (improves match quality)
bad = {"SEE INSTRUMENT", "UNKNOWN", "N/A", ""}
if DEED_NAME_COL in df_deed.columns:
    df_deed = df_deed[~df_deed[DEED_NAME_COL].isin(bad)].copy()
    df_deed = df_deed[df_deed[DEED_NAME_COL].str.len().fillna(0) >= 5].copy()

print("df_deed cols:", df_deed.columns.tolist())
print("df_deed head:\n", df_deed.head(2))

# --- Owner TXT ---
df_owner = pd.read_csv(
    r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\Harris county current owner data 01.29.2026.txt",
    sep="\t",
    dtype=str
)
df_owner.columns = df_owner.columns.str.strip()
df_owner[APP_ACCT_COL] = df_owner[APP_ACCT_COL].astype("string").str.strip()

# Parse purchase_date and filter to last 12 months (business rule)
df_owner[APP_PURCHASE_COL] = pd.to_datetime(df_owner[APP_PURCHASE_COL], errors="coerce")
cutoff = pd.Timestamp.today() - pd.DateOffset(months=12)
df_owner_12mo = df_owner[df_owner[APP_PURCHASE_COL] >= cutoff].copy()

print("df_owner total:", df_owner.shape)
print("df_owner last 12mo:", df_owner_12mo.shape)


# ===========================
# 5) BUILD FEATURES
# ===========================
app_feat = build_features_appraisal(df_owner_12mo)
deed_feat = build_features_deed(df_deed)

print("app_feat:", app_feat.shape, "deed_feat:", deed_feat.shape)
print("Last name overlap:", len(set(app_feat["last_name"]) & set(deed_feat["last_name"])))


# ===========================
# 6) CREATE MATCH INPUT (df_all)
# ===========================
db_api = DuckDBAPI()

app_match = app_feat.copy()
deed_match = deed_feat.copy()

app_match["source_dataset"] = "app"
deed_match["source_dataset"] = "deed"

# unique_id required by Splink
app_match = app_match.reset_index(drop=True).reset_index().rename(columns={"index":"unique_id"})
app_match["unique_id"] = "app_" + app_match["unique_id"].astype(str)

deed_match = deed_match.reset_index(drop=True).reset_index().rename(columns={"index":"unique_id"})
deed_match["unique_id"] = "deed_" + deed_match["unique_id"].astype(str)

# Align schemas for link_only union inside Splink
common_cols = sorted(set(app_match.columns) & set(deed_match.columns))
app_match = app_match[common_cols]
deed_match = deed_match[common_cols]

df_all = pd.concat([app_match, deed_match], ignore_index=True)
print("df_all:", df_all.shape)


# ===========================
# 7) SPLINK SETTINGS + TRAIN + PREDICT
# ===========================
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="unique_id",
    source_dataset_column_name="source_dataset",
    comparisons=[
        cl.NameComparison("name_norm"),
        cl.ExactMatch("last_name"),
        cl.ExactMatch("first_name"),
        cl.ExactMatch("mid_init"),
    ],
    blocking_rules_to_generate_predictions=[
        block_on("last_name"),
        block_on("last3"),
        block_on("last3", "first_name"),
    ],
)

# Train on sample (faster) then predict full
app_train = app_match.sample(n=min(200_000, len(app_match)), random_state=42)
deed_train = deed_match  # deed is small

df_train = pd.concat([app_train, deed_train], ignore_index=True)

linker_train = Linker(df_train, settings, db_api=db_api)

# Train m (EM) and u (random sampling) on sample
linker_train.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule="l.last3 = r.last3"
)
linker_train.training.estimate_u_using_random_sampling(max_pairs=2_000_000)

# Predict on full
linker = Linker(df_all, settings, db_api=db_api)
df_pred = linker.inference.predict().as_pandas_dataframe()

# app -> deed only
df_ad = df_pred[
    (df_pred["source_dataset_l"]=="app") &
    (df_pred["source_dataset_r"]=="deed")
].copy()

print(df_ad["match_probability"].describe(percentiles=[.9,.95,.99,.995,.999]))
print("df_pred rows:", len(df_pred))
print("df_ad rows:", len(df_ad))
print("df_ad p>=0.99:", (df_ad["match_probability"] >= 0.99).sum())

print(df_ad.sort_values("match_probability", ascending=False).head(10)[
    ["unique_id_l","unique_id_r","match_probability","name_norm_l","name_norm_r"]
])


# ===========================
# 8) ACCEPT TOP-1 MATCH PER OWNER + ENRICH WITH ADDRESS + EXPORT
# ===========================
df_ad = df_ad.sort_values(["unique_id_l", "match_probability"], ascending=[True, False])
top1 = df_ad.groupby("unique_id_l").head(1).copy()
accepted = top1[top1["match_probability"] >= 0.99].copy()
print("Accepted matches:", len(accepted))

# Build raw lookup tables with stable unique_ids matching the ones used in Splink
# IMPORTANT: these must match app_match/deed_match unique_id assignment (reset_index order)
app_raw = df_owner_12mo.reset_index(drop=True).reset_index().rename(columns={"index":"row_id"})
app_raw["unique_id"] = "app_" + app_raw["row_id"].astype(str)

deed_raw = df_deed.reset_index(drop=True).reset_index().rename(columns={"index":"row_id"})
deed_raw["unique_id"] = "deed_" + deed_raw["row_id"].astype(str)

owner_lookup = app_raw[[
    "unique_id", APP_ACCT_COL, APP_OWNER_COL, APP_ADDR_COL, APP_PURCHASE_COL
]].rename(columns={
    "unique_id": "unique_id_l",
    APP_ACCT_COL: "acct",
    APP_OWNER_COL: "owner_name",
    APP_ADDR_COL: "site_address",
    APP_PURCHASE_COL: "purchase_date"
})

deed_keep = [c for c in [DEED_FILE_NO_COL, DEED_FILEDATE_COL, DEED_NAME_COL, DEED_SUBDIV_COL, DEED_SECTION_COL, DEED_LOT_COL, DEED_BLOCK_COL, DEED_UNIT_COL] if c in deed_raw.columns]
deed_lookup = deed_raw[["unique_id"] + deed_keep].rename(columns={"unique_id":"unique_id_r"})

accepted_enriched = (
    accepted
    .merge(owner_lookup, on="unique_id_l", how="left")
    .merge(deed_lookup, on="unique_id_r", how="left")
)

# Ensure purchase_date is datetime and enforce 12-month rule at export too (belt + suspenders)
accepted_enriched["purchase_date"] = pd.to_datetime(accepted_enriched["purchase_date"], errors="coerce")
cutoff = pd.Timestamp.today() - pd.DateOffset(months=12)
accepted_enriched = accepted_enriched[accepted_enriched["purchase_date"] >= cutoff].copy()

# Put important cols up front
front = ["File No","FileDate","Name","acct","owner_name","site_address","purchase_date","match_probability"]
front = [c for c in front if c in accepted_enriched.columns]
accepted_enriched = accepted_enriched[front + [c for c in accepted_enriched.columns if c not in front]]

out_path = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\deed enriched with date filter\deed_enriched_with_owner_addressdatef.csv"
accepted_enriched.to_csv(out_path, index=False)

print("Wrote rows:", len(accepted_enriched))
print("Output:", out_path)
print("Address null %:", accepted_enriched["site_address"].isna().mean() if "site_address" in accepted_enriched.columns else "no address col")


df_deed cols: ['File No', 'SubDivision', 'Section', 'Lot', 'Block', 'Misc', 'Map Page', 'Unit', 'Abstract', 'OutLot', 'Tract', 'Reserve', 'Comment', 'Document Type', 'FileDate', 'Film Code No.', 'No. of Pages', 'Name']
df_deed head:
           File No       SubDivision Section   Lot Block  Misc Map Page  Unit  \
1  RP-2025-309774  SHADYVILLA HOMES    <NA>  <NA>  <NA>  <NA>     <NA>  <NA>   
2  RP-2025-309775    THORNTON VISTA    <NA>  <NA>  <NA>  <NA>     <NA>  <NA>   

  Abstract OutLot Tract Reserve Comment Document Type    FileDate  \
1     <NA>   <NA>  <NA>    <NA>    <NA>           MAP  2025-08-07   
2     <NA>   <NA>  <NA>    <NA>    <NA>           MAP  2025-08-07   

    Film Code No. No. of Pages                  Name  
1  RP-2025-309774            1    COGSWELL E B A0785  
2  RP-2025-309775            1  ALLEN SAMUEL W A0094  
df_owner total: (1601712, 4)
df_owner last 12mo: (67393, 4)
app_feat: (67393, 17) deed_feat: (1757, 26)
Last name overlap: 751
df_all: (69150, 9)



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.last3 = r.last3

Parameter estimates will be made for the following comparison(s):
    - name_norm
    - last_name
    - first_name
    - mid_init

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 

Iteration 1: Largest change in params was -0.826 in the m_probability of name_norm, level `Exact match on name_norm`
Iteration 2: Largest change in params was 0.551 in the m_probability of mid_init, level `All other comparisons`
Iteration 3: Largest change in params was 0.27 in the m_probability of name_norm, level `All other comparisons`
Iteration 4: Largest change in params was 0.0838 in the m_probability of last_name, level `All other comparisons`
Iteration 5: Largest change in params was 0.0606 in the m_probability of last_name, level `All other comparisons`
Iteration 6: Largest change in params was -0.0193 in the m_probabi

count    2.494340e+05
mean     3.970032e-02
std      1.751319e-01
min      9.537697e-11
50%      9.537697e-11
90%      1.984410e-05
95%      3.940365e-01
99%      9.999988e-01
99.5%    9.999988e-01
99.9%    1.000000e+00
max      1.000000e+00
Name: match_probability, dtype: float64
df_pred rows: 249434
df_ad rows: 249434
df_ad p>=0.99: 4767
       unique_id_l unique_id_r  match_probability  \
142064   app_50806   deed_1399                1.0   
171848    app_9324   deed_1559                1.0   
172862   app_13688    deed_782                1.0   
159125   app_47600    deed_275                1.0   
137188   app_34847    deed_686                1.0   
167229   app_41798    deed_109                1.0   
167234   app_32330    deed_757                1.0   
187865   app_45305    deed_102                1.0   
187864   app_26634     deed_48                1.0   
167227   app_28229   deed_1243                1.0   

                          name_norm_l                    name_norm_r  
142